In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor, XGBClassifier

# ==========================================
# 1. LOAD PREPROCESSED DATASET
# ==========================================
df = pd.read_csv('df_EDA_standard.csv')

# Separate features and target spaces
X = df.drop(columns=['SalePrice', 'Price_Category'], errors='ignore')
y_reg = df['SalePrice']
y_clf = df['Price_Category']

print(f"Data ready for XGBoost. Features shape: {X.shape}")

# ==========================================
# 2. XGBOOST REGRESSOR (Continuous SalePrice)
# ==========================================
print("\n--- Tuning XGBoost Regressor ---")

# Define an efficient hyperparameter grid for regression
xgb_reg_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

xgb_reg = XGBRegressor(random_state=42, objective='reg:squarederror')

grid_xgb_reg = GridSearchCV(
    estimator=xgb_reg,
    param_grid=xgb_reg_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_xgb_reg.fit(X, y_reg)

best_rmse = np.sqrt(-grid_xgb_reg.best_score_)
print(f"Best Regressor Parameters: {grid_xgb_reg.best_params_}")
print(f"Optimized XGBoost Regressor Validation RMSE: ${best_rmse:,.2f}")


# ==========================================
# 3. XGBOOST CLASSIFIER (Categorical Tiers)
# ==========================================
print("\n--- Tuning XGBoost Classifier ---")

# Convert categorical labels to integers (0, 1, 2) to comply with XGBoost rules
le = LabelEncoder()
y_clf_encoded = le.fit_transform(y_clf)

# Define hyperparameter grid for multi-class classification
xgb_clf_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'gamma': [0, 0.1]
}

# multi:softprob outputs probabilities for each category tier
xgb_clf = XGBClassifier(random_state=42, objective='multi:softprob', eval_metric='mlogloss')

grid_xgb_clf = GridSearchCV(
    estimator=xgb_clf,
    param_grid=xgb_clf_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid_xgb_clf.fit(X, y_clf_encoded)

print(f"Best Classifier Parameters: {grid_xgb_clf.best_params_}")
print(f"Optimized XGBoost Classifier Validation Macro F1-Score: {grid_xgb_clf.best_score_:.4f}")

Data ready for XGBoost. Features shape: (1460, 224)

--- Tuning XGBoost Regressor ---
Best Regressor Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Optimized XGBoost Regressor Validation RMSE: $26,116.06

--- Tuning XGBoost Classifier ---
Best Classifier Parameters: {'gamma': 0, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Optimized XGBoost Classifier Validation Macro F1-Score: 0.8702


In [2]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
    --------------------------------------- 1.0/69.5 MB 11.2 MB/s eta 0:00:07
   - -------------------------------------- 3.4/69.5 MB 11.5 MB/s eta 0:00:06
   -- ------------------------------------- 5.0/69.5 MB 9.2 MB/s eta 0:00:08
   --- ------------------------------------ 6.6/69.5 MB 8.9 MB/s eta 0:00:08
   ---- ----------------------------------- 8.4/69.5 MB 8.8 MB/s eta 0:00:07
   ----- ---------------------------------- 10.0/69.5 MB 8.7 MB/s eta 0:00:07
   ------ --------------------------------- 12.1/69.5 MB 8.7 MB/s eta 0:00:07
   ------- -------------------------------- 13.9/69.5 MB 8.6 MB/s eta 0:00:07
   --------- ------------------------------ 15.7/69.5 MB 8.6 MB/s eta 0:00:07
   ---------- ----------------------------- 17.6/69.5 MB 8.6 MB/s eta 0:00:07
   ----------- ---------------------------- 19.4/69.5 MB 8.7 MB/s eta 0:00:


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
